**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../docs/METHODS.md) for the implemented protocol.


# 02 V2 — Preprocessing for Level 2 with chroma replaced

Re-runs the preprocessing pipeline (resample → 2-sec windows → bandpass)
for the Level 2 clip subset only, using
`v2.utils.features.extract_features_level2_v2` instead of the V1
`extract_features_level2`. The 12 chroma dims are replaced with 6 MFCC
std + 6 delta-MFCC std (Fix 6).

Output: `v2/Level2/Data/level2_features_v2.npz`. Levels 1 and 3 are not
regenerated (Fix 6 only changes Level 2 features).


In [ ]:
import io
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from datasets import load_from_disk, Audio

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.preprocessing import preprocess_clip, TARGET_SR
from v2.utils.features import extract_features_level2_v2

HF_LOCAL = PROJECT_ROOT / "Mss" / "data" / "wmms_parquet_local"
METADATA_CSV = PROJECT_ROOT / "wmmd_metadata.csv"
OUT_DIR = PROJECT_ROOT / "v2" / "Level2" / "Data"
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Load dataset and pick the same Level 2 clip subset as V1


In [ ]:
print("Loading dataset...")
ds = load_from_disk(str(HF_LOCAL))
ds = ds.cast_column("audio", Audio(decode=False))
meta = pd.read_csv(METADATA_CSV)
print(f"  train={len(ds['train'])}, test={len(ds['test'])}")

l2_meta = meta[meta["family"] == "Delphinidae"].copy()
keep_l2 = set(l2_meta.index.tolist())
label_names_l2 = sorted(meta[meta["family"] == "Delphinidae"]["species"].unique().tolist())
l2_label_to_idx = {s: i for i, s in enumerate(label_names_l2)}
print(f"Level 2: {len(keep_l2)} clips, {len(label_names_l2)} species")


## Stream clips, preprocess, extract V2 features


In [ ]:
def iter_clips(ds, meta):
    lookup = {}
    for gid, row in meta.iterrows():
        lookup[(row["split"], int(row["index"]))] = (
            int(gid), str(row["species"]), str(row["family"])
        )
    for split_name in ("train", "test"):
        split = ds[split_name]
        for i in range(len(split)):
            sample = split[i]
            data, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))
            if data.ndim > 1:
                data = data.mean(axis=1)
            data = np.asarray(data, dtype=np.float32)
            gid, species, family = lookup[(split_name, i)]
            yield {
                "global_clip_id": gid, "species": species, "family": family,
                "sampling_rate": int(sr), "audio": data,
                "is_train": split_name == "train",
            }

X_list, labels_list, clip_ids_list = [], [], []
species_list, is_train_list = [], []

t0 = time.time()
n_clips_done = 0
for clip in iter_clips(ds, meta):
    gid = clip["global_clip_id"]
    if gid not in keep_l2:
        continue
    n_clips_done += 1
    windows = preprocess_clip(clip["audio"], orig_sr=clip["sampling_rate"])
    for w in windows:
        feat = extract_features_level2_v2(w)
        X_list.append(feat)
        labels_list.append(l2_label_to_idx[clip["species"]])
        clip_ids_list.append(gid)
        species_list.append(clip["species"])
        is_train_list.append(clip["is_train"])
    if n_clips_done % 200 == 0:
        print(f"  processed {n_clips_done}/{len(keep_l2)} clips, {time.time()-t0:.1f}s")

print(f"\nTotal time: {time.time() - t0:.1f}s")

X = np.stack(X_list).astype(np.float32)
labels = np.array(labels_list, dtype=np.int64)
clip_ids = np.array(clip_ids_list, dtype=np.int64)
species = np.array(species_list, dtype=object)
is_train = np.array(is_train_list, dtype=bool)
print(f"X: {X.shape}, labels: {labels.shape}")

# Sanity check vs V1: same N, same is_train counts.
v1 = np.load(PROJECT_ROOT / "Level2" / "Data" / "level2_features.npz", allow_pickle=True)
print(f"V1 X shape: {v1['X'].shape}, V2 X shape: {X.shape}")
assert X.shape[0] == v1["X"].shape[0], "window count mismatch with V1"
assert is_train.sum() == v1["is_train"].sum(), "is_train count mismatch"

np.savez_compressed(
    OUT_DIR / "level2_features_v2.npz",
    X=X, labels=labels, clip_ids=clip_ids,
    label_names=np.array(label_names_l2, dtype=object),
    species=species, is_train=is_train,
)
print(f"wrote {OUT_DIR / 'level2_features_v2.npz'}")
